# CABIN-EXP-020A — Cabin / Driver View Baseline

Amaç: `CABIN-EXP-012` heuristik ROI yaklaşımı reddedildiği için model-first bir **cabin/driver görünürlük gate** baseline'ı kurmak.

Bu notebook doğrudan `telefonla_konusma`, `su_icme`, `arkaya_bakma`, `etrafa_bakinma` kararı vermez. Önce şu soruyu çözer:

```text
Bu frame/crop içinde sürücü/cabin görünür mü, yoksa dış yol/araç görüntüsü mü?
```

İlk sınıflar:

```text
driver_cabin_visible
not_cabin_view
```

Veri kaynakları:

1. **State Farm Distracted Driver Detection** — driver/cabin görünür pozitif görüntüler ve FTR'ye yakın action sınıfları.
   Kaynak: https://www.kaggle.com/competitions/state-farm-distracted-driver-detection/data
2. **BDD100K veya manuel negatif klasör** — dış yol/araç/cabin olmayan negatif örnekler.
   Negatif klasör varsayılanı: `/content/drive/MyDrive/anomali-road-safety-ai/datasets/cabin_exp_020a/negatives/not_cabin_view/`
3. **AUC Distracted Driver Dataset** — bu notebook'ta opsiyonel/manual input olarak bırakılır.
   Kaynak: https://heshameraqi.github.io/distraction_detection

Neden State Farm?

Kaggle sayfasında 10 sınıf bulunur: `c0 safe driving`, `c1/c3 texting`, `c2/c4 talking on the phone`, `c6 drinking`, `c7 reaching behind`, `c9 talking to passenger`. Bu sınıflar sonraki `CABIN-EXP-020B` action classifier için de aynı veri omurgasını kullanmamızı sağlar.

Önemli sınırlama:

State Farm/AUC genellikle araç içi kamera verisidir. Bizim demo dış kameradan yan/ön cam görür. Bu nedenle 020A bir domain-adaptation başlangıcıdır; kesin saha performansı iddiası değildir.

In [ ]:
# Cell 1 — Runtime setup
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

def run(cmd, check=True):
    print('+', ' '.join(map(str, cmd)))
    return subprocess.run(cmd, check=check)

if IN_COLAB:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle', 'timm', 'scikit-learn', 'seaborn', 'opencv-python', 'tqdm'])

import json
import math
import random
import shutil
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('IN_COLAB:', IN_COLAB)
print('DEVICE:', DEVICE)
print('torch:', torch.__version__)

In [ ]:
# Cell 2 — Drive mount + configuration
from pathlib import Path

MOUNT_DRIVE = True
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai') if IN_COLAB else Path.cwd()
WORK_ROOT = Path('/content/anomali-road-safety-ai-work') if IN_COLAB else Path.cwd() / '.tmp_cabin_exp_020a_work'
WORK_ROOT.mkdir(parents=True, exist_ok=True)

EXP_ID = 'CABIN-EXP-020A'
DATA_ROOT = PROJECT_ROOT / 'datasets' / 'cabin_exp_020a'
STATE_FARM_ROOT = DATA_ROOT / 'state_farm'
NEGATIVE_ROOT = DATA_ROOT / 'negatives' / 'not_cabin_view'
AUC_ROOT = DATA_ROOT / 'auc_distracted_driver_optional'
OUT_ROOT = PROJECT_ROOT / 'models' / 'benchmarks' / 'artifacts' / 'cabin_driver' / EXP_ID
CKPT_ROOT = PROJECT_ROOT / 'models' / 'checkpoints' / 'cabin_driver' / EXP_ID
REPORT_ROOT = PROJECT_ROOT / 'testing' / 'reports'
LOCAL_DATA_ROOT = WORK_ROOT / 'datasets' / 'cabin_exp_020a'
LOCAL_STATE_FARM_ROOT = LOCAL_DATA_ROOT / 'state_farm'
LOCAL_NEGATIVE_ROOT = LOCAL_DATA_ROOT / 'negatives' / 'not_cabin_view'

for p in [DATA_ROOT, STATE_FARM_ROOT, NEGATIVE_ROOT, AUC_ROOT, OUT_ROOT, CKPT_ROOT, REPORT_ROOT, LOCAL_DATA_ROOT, LOCAL_STATE_FARM_ROOT, LOCAL_NEGATIVE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Kaggle competition source. Requires Kaggle credentials and accepting competition terms.
STATE_FARM_COMPETITION = 'state-farm-distracted-driver-detection'
# Competition download is preferred. If it fails because rules are not accepted or
# Kaggle returns 403/404, the notebook can try public dataset mirrors.
# Leave empty if you only want the official competition source.
KAGGLE_DATASET_FALLBACKS = [
    # Example format: 'owner/dataset-slug'
    # Add a trusted mirror here only if the official competition API is unavailable.
]
DOWNLOAD_STATE_FARM = True
USE_EXISTING_STATE_FARM = True

# Negative data strategy:
# 1) If NEGATIVE_ROOT contains jpg/png images, use them.
# 2) Else, sample BDD100K images if found in Drive.
# 3) Else fail unless ALLOW_POSITIVE_ONLY_SMOKE=True.
USE_MANUAL_NEGATIVES = True
USE_BDD100K_NEGATIVES = True
ALLOW_POSITIVE_ONLY_SMOKE = False
BDD_CANDIDATE_ROOTS = [
    PROJECT_ROOT / 'datasets' / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'train',
    PROJECT_ROOT / 'datasets' / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'val',
    PROJECT_ROOT / 'datasets' / 'bdd100k' / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'train',
    PROJECT_ROOT / 'datasets' / 'bdd100k' / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'val',
]

# Training controls
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 2
EPOCHS = 8
PATIENCE = 3
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
BACKBONES = ['mobilenet_v3_large', 'efficientnet_b0']

# Keep this bounded for first Colab run. Set None for full State Farm positives.
MAX_POSITIVE_IMAGES = 12000
MAX_NEGATIVE_IMAGES = 12000
MIN_NEGATIVE_IMAGES_REQUIRED = 500
TEST_SIZE = 0.15
VAL_SIZE = 0.15

LABELS = ['driver_cabin_visible', 'not_cabin_view']
LABEL_TO_ID = {name: idx for idx, name in enumerate(LABELS)}
ID_TO_LABEL = {idx: name for name, idx in LABEL_TO_ID.items()}

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORK_ROOT:', WORK_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('CKPT_ROOT:', CKPT_ROOT)

In [ ]:
# Cell 3 — Kaggle auth + download helpers
import os
import json
from pathlib import Path

def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

def count_images(root: Path):
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

def ensure_kaggle_credentials():
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / 'kaggle.json'

    secret_name_pairs = [
        ('KAGGLE_USERNAME', 'KAGGLE_KEY'),
        ('kaggle_username', 'kaggle_key'),
        ('KAGGLE_USER', 'KAGGLE_TOKEN'),
        ('kaggle_user', 'kaggle_token'),
    ]

    def write_credentials(username, key, source_label):
        username = str(username).strip()
        key = str(key).strip()
        if not username or not key:
            return False
        target.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
        os.chmod(target, 0o600)
        print(f'Kaggle credentials from {source_label}:', target)
        return True

    # 1) Colab Secrets, if available. Accept both uppercase and lowercase names.
    if IN_COLAB:
        try:
            from google.colab import userdata
            for username_name, key_name in secret_name_pairs:
                username = userdata.get(username_name)
                key = userdata.get(key_name)
                if username and key:
                    return write_credentials(username, key, f'Colab Secrets ({username_name}/{key_name})')
        except Exception as exc:
            print('Colab Secrets unavailable/skipped:', exc)

    # 2) Environment variables. Accept both uppercase and lowercase names.
    for username_name, key_name in secret_name_pairs:
        username = os.environ.get(username_name)
        key = os.environ.get(key_name)
        if username and key:
            return write_credentials(username, key, f'environment ({username_name}/{key_name})')

    # 3) Existing kaggle.json files.
    candidates = [
        PROJECT_ROOT / 'kaggle.json',
        Path('/content/kaggle.json'),
        target,
    ]
    for c in candidates:
        if c.exists():
            if c != target:
                shutil.copy2(c, target)
            os.chmod(target, 0o600)
            print('Kaggle credentials:', target)
            return True

    # 4) Interactive upload fallback. Do not save credentials into Git.
    if IN_COLAB:
        print('Kaggle credentials not found. Upload kaggle.json now if needed.')
        try:
            from google.colab import files
            uploaded = files.upload()
            if 'kaggle.json' in uploaded:
                Path('/content/kaggle.json').write_bytes(uploaded['kaggle.json'])
                shutil.copy2('/content/kaggle.json', target)
                os.chmod(target, 0o600)
                print('Kaggle credentials uploaded:', target)
                return True
        except Exception as exc:
            print('Upload skipped/failed:', exc)
    return False

def unzip_once(zip_path: Path, extract_dir: Path, marker_name: str | None = None):
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    marker = extract_dir / (marker_name or (zip_path.name + '.extracted.marker'))
    if marker.exists():
        print('Already extracted:', zip_path)
        return
    extract_dir.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_path, '->', extract_dir)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    marker.write_text(now_utc())

def run_kaggle_command(cmd):
    print('+', ' '.join(map(str, cmd)))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print('--- kaggle stdout ---')
        print(result.stdout[-4000:])
    if result.stderr:
        print('--- kaggle stderr ---')
        print(result.stderr[-4000:])
    if result.returncode != 0:
        raise RuntimeError(
            'Kaggle command failed with exit code ' + str(result.returncode) + '\n'
            'Most common causes for this notebook:\n'
            '1) You have not accepted the State Farm competition rules on Kaggle. Open the data page and click Join/Accept.\n'
            '2) The Kaggle account in Colab Secrets is different from the account that accepted the rules.\n'
            '3) KAGGLE_USERNAME/KAGGLE_KEY is invalid, revoked, copied with spaces, or belongs to another account.\n'
            '4) The old competition API is temporarily unavailable. In that case, use KAGGLE_DATASET_FALLBACKS or manual zip.\n'
            'Official data page: https://www.kaggle.com/competitions/state-farm-distracted-driver-detection/data'
        )
    return result

def normalize_state_farm_layout():
    train_dir = STATE_FARM_ROOT / 'imgs' / 'train'
    if train_dir.exists() and count_images(train_dir) > 1000:
        return train_dir

    # Official competition often extracts imgs.zip plus csv files.
    imgs_zip = STATE_FARM_ROOT / 'imgs.zip'
    if imgs_zip.exists():
        unzip_once(imgs_zip, STATE_FARM_ROOT, marker_name='imgs_zip.extracted.marker')

    train_dir = STATE_FARM_ROOT / 'imgs' / 'train'
    if train_dir.exists() and count_images(train_dir) > 1000:
        return train_dir

    # Some mirrors extract directly as train/c0..c9.
    alt = STATE_FARM_ROOT / 'train'
    if alt.exists() and count_images(alt) > 1000:
        (STATE_FARM_ROOT / 'imgs').mkdir(exist_ok=True)
        if not train_dir.exists():
            shutil.move(str(alt), str(train_dir))
        return train_dir

    # Some mirrors extract nested folders. Search shallowly for a train directory with c0..c9.
    candidates = []
    for candidate in STATE_FARM_ROOT.rglob('train'):
        if candidate.is_dir() and all((candidate / f'c{i}').exists() for i in range(10)):
            candidates.append(candidate)
    if candidates:
        src = sorted(candidates, key=lambda x: count_images(x), reverse=True)[0]
        train_dir.parent.mkdir(parents=True, exist_ok=True)
        if src != train_dir and not train_dir.exists():
            shutil.copytree(src, train_dir)
        return train_dir
    return train_dir

def download_state_farm():
    existing_train = STATE_FARM_ROOT / 'imgs' / 'train'
    if USE_EXISTING_STATE_FARM and count_images(existing_train) > 1000:
        print('State Farm already prepared in Drive:', existing_train, 'images:', count_images(existing_train))
        return
    if not DOWNLOAD_STATE_FARM:
        raise FileNotFoundError('State Farm data not found and DOWNLOAD_STATE_FARM=False')
    if not ensure_kaggle_credentials():
        raise RuntimeError('Kaggle credentials are required. Put kaggle.json under PROJECT_ROOT or upload it in Colab.')

    zip_path = STATE_FARM_ROOT / f'{STATE_FARM_COMPETITION}.zip'
    competition_error = None
    if not zip_path.exists():
        try:
            run_kaggle_command(['kaggle', 'competitions', 'download', '-c', STATE_FARM_COMPETITION, '-p', str(STATE_FARM_ROOT)])
        except Exception as exc:
            competition_error = exc
            print('Official competition download failed. Will try configured dataset fallbacks if any.')
            print(str(exc))
        downloaded = STATE_FARM_ROOT / f'{STATE_FARM_COMPETITION}.zip'
        if downloaded.exists() and downloaded != zip_path:
            downloaded.rename(zip_path)

    if zip_path.exists():
        unzip_once(zip_path, STATE_FARM_ROOT, marker_name='competition_zip.extracted.marker')
    else:
        for dataset_slug in KAGGLE_DATASET_FALLBACKS:
            safe_slug = dataset_slug.replace('/', '__')
            fallback_dir = STATE_FARM_ROOT / f'fallback_{safe_slug}'
            fallback_dir.mkdir(parents=True, exist_ok=True)
            try:
                run_kaggle_command(['kaggle', 'datasets', 'download', '-d', dataset_slug, '-p', str(fallback_dir), '--unzip'])
                # Copy extracted content into STATE_FARM_ROOT for layout normalization.
                for child in fallback_dir.iterdir():
                    target = STATE_FARM_ROOT / child.name
                    if target.exists():
                        continue
                    if child.is_dir():
                        shutil.copytree(child, target)
                    else:
                        shutil.copy2(child, target)
                break
            except Exception as exc:
                print('Fallback failed:', dataset_slug, exc)

    train_dir = normalize_state_farm_layout()
    if count_images(train_dir) < 1000:
        msg = (
            'State Farm data was not prepared. Expected imgs/train/c0..c9 under: ' + str(STATE_FARM_ROOT) + '\n'
            'Fix options:\n'
            '1) Open https://www.kaggle.com/competitions/state-farm-distracted-driver-detection/data and accept/join the competition, then rerun Cell 3.\n'
            '2) Confirm Colab Secrets KAGGLE_USERNAME/KAGGLE_KEY belong to that same Kaggle account.\n'
            '3) Manually place imgs.zip or extracted imgs/train/c0..c9 under the State Farm Drive folder.\n'
            '4) Add a trusted Kaggle dataset mirror slug to KAGGLE_DATASET_FALLBACKS in Cell 2.\n'
        )
        if competition_error:
            msg += '\nOriginal competition error:\n' + str(competition_error)
        raise RuntimeError(msg)
    print('State Farm train images:', count_images(train_dir), train_dir)

download_state_farm()

In [ ]:
# Cell 4 — Resolve State Farm positives and negative images
IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}
STATE_FARM_CLASS_MAP = {
    'c0': 'safe_driving',
    'c1': 'texting_right',
    'c2': 'talking_phone_right',
    'c3': 'texting_left',
    'c4': 'talking_phone_left',
    'c5': 'operating_radio',
    'c6': 'drinking',
    'c7': 'reaching_behind',
    'c8': 'hair_and_makeup',
    'c9': 'talking_to_passenger',
}

def image_files(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTS])

state_train_dir = STATE_FARM_ROOT / 'imgs' / 'train'
state_driver_csv = STATE_FARM_ROOT / 'driver_imgs_list.csv'
if not state_train_dir.exists():
    raise FileNotFoundError('State Farm train directory not found: ' + str(state_train_dir))

driver_lookup = {}
if state_driver_csv.exists():
    driver_df = pd.read_csv(state_driver_csv)
    for row in driver_df.itertuples(index=False):
        driver_lookup[str(row.img)] = str(row.subject)
    print('driver_imgs_list rows:', len(driver_lookup))
else:
    print('driver_imgs_list.csv missing; will group by class/file hash fallback.')

positive_rows = []
for cls_dir in sorted(state_train_dir.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls = cls_dir.name
    for img_path in image_files(cls_dir):
        positive_rows.append({
            'path': str(img_path),
            'label': 'driver_cabin_visible',
            'label_id': LABEL_TO_ID['driver_cabin_visible'],
            'source': 'state_farm',
            'source_class': cls,
            'source_action': STATE_FARM_CLASS_MAP.get(cls, cls),
            'group': driver_lookup.get(img_path.name, f'unknown_driver_{cls}'),
        })

if MAX_POSITIVE_IMAGES and len(positive_rows) > MAX_POSITIVE_IMAGES:
    rng = random.Random(SEED)
    # Preserve class balance as much as possible.
    by_cls = defaultdict(list)
    for r in positive_rows:
        by_cls[r['source_class']].append(r)
    keep = []
    per_cls = max(1, MAX_POSITIVE_IMAGES // max(1, len(by_cls)))
    for cls, rows in by_cls.items():
        rng.shuffle(rows)
        keep.extend(rows[:per_cls])
    positive_rows = keep[:MAX_POSITIVE_IMAGES]

print('positive rows:', len(positive_rows), Counter(r['source_class'] for r in positive_rows))

def collect_manual_negatives():
    rows = []
    for p in image_files(NEGATIVE_ROOT):
        rows.append({
            'path': str(p),
            'label': 'not_cabin_view',
            'label_id': LABEL_TO_ID['not_cabin_view'],
            'source': 'manual_negative',
            'source_class': 'not_cabin_view',
            'source_action': 'not_cabin_view',
            'group': f'manual_neg_{p.parent.name}',
        })
    return rows

def resolve_bdd_image_dirs():
    dirs = []
    for root in BDD_CANDIDATE_ROOTS:
        n = count_images(root)
        if n > 0:
            dirs.append((root, n))
    return sorted(dirs, key=lambda x: x[1], reverse=True)

def collect_bdd_negatives(max_count):
    dirs = resolve_bdd_image_dirs()
    if not dirs:
        return []
    print('BDD negative candidate dirs:')
    for d, n in dirs[:5]:
        print(' ', d, n)
    all_imgs = []
    for d, _ in dirs:
        all_imgs.extend(image_files(d))
    rng = random.Random(SEED)
    rng.shuffle(all_imgs)
    rows = []
    for p in all_imgs[:max_count]:
        rows.append({
            'path': str(p),
            'label': 'not_cabin_view',
            'label_id': LABEL_TO_ID['not_cabin_view'],
            'source': 'bdd100k_negative',
            'source_class': 'not_cabin_view',
            'source_action': 'not_cabin_view',
            'group': f'bdd_{p.stem[:8]}',
        })
    return rows

negative_rows = []
if USE_MANUAL_NEGATIVES:
    negative_rows.extend(collect_manual_negatives())
if USE_BDD100K_NEGATIVES and len(negative_rows) < MIN_NEGATIVE_IMAGES_REQUIRED:
    target_neg = MAX_NEGATIVE_IMAGES or len(positive_rows)
    negative_rows.extend(collect_bdd_negatives(target_neg))

# Dedupe paths.
seen = set()
negative_rows = [r for r in negative_rows if not (r['path'] in seen or seen.add(r['path']))]
if MAX_NEGATIVE_IMAGES and len(negative_rows) > MAX_NEGATIVE_IMAGES:
    random.Random(SEED).shuffle(negative_rows)
    negative_rows = negative_rows[:MAX_NEGATIVE_IMAGES]

if len(negative_rows) < MIN_NEGATIVE_IMAGES_REQUIRED and not ALLOW_POSITIVE_ONLY_SMOKE:
    raise RuntimeError(
        f'Not enough negative images: {len(negative_rows)}. Put not-cabin images under {NEGATIVE_ROOT} '
        'or make sure BDD100K images exist in Drive. Set ALLOW_POSITIVE_ONLY_SMOKE=True only for smoke tests.'
    )

print('negative rows:', len(negative_rows), Counter(r['source'] for r in negative_rows))
metadata_df = pd.DataFrame(positive_rows + negative_rows)
metadata_path = OUT_ROOT / 'cabin_exp_020a_metadata.csv'
metadata_df.to_csv(metadata_path, index=False)
print('metadata:', metadata_path, metadata_df.shape)
display(metadata_df.groupby(['label', 'source']).size().reset_index(name='count'))

In [ ]:
# Cell 5 — Build train/val/test split with driver-group protection for positives
from sklearn.model_selection import GroupShuffleSplit, train_test_split

pos_df = metadata_df[metadata_df['label'] == 'driver_cabin_visible'].copy()
neg_df = metadata_df[metadata_df['label'] == 'not_cabin_view'].copy()

def split_positive_by_driver(df):
    groups = df['group'].fillna('unknown').astype(str).values
    gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    train_val_idx, test_idx = next(gss1.split(df, df['label_id'], groups))
    train_val = df.iloc[train_val_idx].copy()
    test = df.iloc[test_idx].copy()
    val_fraction_of_train_val = VAL_SIZE / (1.0 - TEST_SIZE)
    groups_tv = train_val['group'].fillna('unknown').astype(str).values
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction_of_train_val, random_state=SEED + 1)
    train_idx, val_idx = next(gss2.split(train_val, train_val['label_id'], groups_tv))
    train = train_val.iloc[train_idx].copy()
    val = train_val.iloc[val_idx].copy()
    return train, val, test

def split_negatives(df):
    if len(df) == 0:
        return df.copy(), df.copy(), df.copy()
    train_val, test = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, shuffle=True)
    val_fraction_of_train_val = VAL_SIZE / (1.0 - TEST_SIZE)
    train, val = train_test_split(train_val, test_size=val_fraction_of_train_val, random_state=SEED + 1, shuffle=True)
    return train.copy(), val.copy(), test.copy()

pos_train, pos_val, pos_test = split_positive_by_driver(pos_df)
neg_train, neg_val, neg_test = split_negatives(neg_df)

for name, df in [('train', pos_train), ('val', pos_val), ('test', pos_test)]:
    print(name, 'positive drivers:', df['group'].nunique(), 'rows:', len(df))

train_df = pd.concat([pos_train, neg_train], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat([pos_val, neg_val], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat([pos_test, neg_test], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

for split_name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df['split'] = split_name
    path = OUT_ROOT / f'cabin_exp_020a_{split_name}.csv'
    df.to_csv(path, index=False)
    print(split_name, df.shape, Counter(df['label']))

split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_path = OUT_ROOT / 'cabin_exp_020a_splits.csv'
split_df.to_csv(split_path, index=False)

# Leakage check for State Farm drivers.
positive_split = split_df[split_df['source'] == 'state_farm']
leak = positive_split.groupby('group')['split'].nunique()
leaky_groups = leak[leak > 1]
print('positive driver leakage groups:', len(leaky_groups))
if len(leaky_groups):
    raise RuntimeError('Driver leakage detected across splits: ' + ', '.join(leaky_groups.index[:10]))

display(split_df.groupby(['split', 'label', 'source']).size().reset_index(name='count'))

In [ ]:
# Cell 6 — Dataset and dataloaders
class CabinViewDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['label_id'])
        return img, label

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.03),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = CabinViewDataset(train_df, train_tfms)
val_ds = CabinViewDataset(val_df, eval_tfms)
test_ds = CabinViewDataset(test_df, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(len(train_ds), len(val_ds), len(test_ds))

In [ ]:
# Cell 7 — Model helpers
NUM_CLASSES = len(LABELS)

def build_model(backbone):
    if backbone == 'mobilenet_v3_large':
        weights = models.MobileNet_V3_Large_Weights.DEFAULT
        model = models.mobilenet_v3_large(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
        return model
    if backbone == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, NUM_CLASSES)
        return model
    raise ValueError(backbone)

def class_weights_from_df(df):
    counts = Counter(df['label_id'])
    total = sum(counts.values())
    weights = []
    for i in range(NUM_CLASSES):
        weights.append(total / max(1, NUM_CLASSES * counts.get(i, 1)))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    losses = []
    criterion = nn.CrossEntropyLoss(weight=class_weights_from_df(train_df))
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.softmax(logits, dim=1)
        pred = probs.argmax(dim=1)
        losses.append(float(loss.item()))
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        y_prob.extend(probs.cpu().numpy().tolist())
    return {
        'loss': float(np.mean(losses)) if losses else None,
        'accuracy': accuracy_score(y_true, y_pred) if y_true else None,
        'macro_f1': f1_score(y_true, y_pred, average='macro') if y_true else None,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

def train_one_backbone(backbone):
    model = build_model(backbone).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_from_df(train_df))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))
    best_f1 = -1.0
    best_path = CKPT_ROOT / f'{EXP_ID}-{backbone}-best.pth'
    history = []
    patience_left = PATIENCE
    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for images, labels in tqdm(train_loader, desc=f'{backbone} epoch {epoch}/{EPOCHS}'):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))
        scheduler.step()
        val_metrics = evaluate(model, val_loader)
        row = {
            'epoch': epoch,
            'train_loss': float(np.mean(train_losses)) if train_losses else None,
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
            'val_macro_f1': val_metrics['macro_f1'],
        }
        history.append(row)
        print(row)
        score = val_metrics['macro_f1'] or 0.0
        if score > best_f1:
            best_f1 = score
            patience_left = PATIENCE
            torch.save({
                'experiment_id': EXP_ID,
                'backbone': backbone,
                'state_dict': model.state_dict(),
                'labels': LABELS,
                'label_to_id': LABEL_TO_ID,
                'img_size': IMG_SIZE,
                'created_at_utc': now_utc(),
            }, best_path)
            print('saved best:', best_path)
        else:
            patience_left -= 1
            if patience_left <= 0:
                print('early stopping')
                break
    ckpt = torch.load(best_path, map_location=DEVICE)
    model = build_model(backbone).to(DEVICE)
    model.load_state_dict(ckpt['state_dict'])
    val_metrics = evaluate(model, val_loader)
    test_metrics = evaluate(model, test_loader)
    return model, {
        'backbone': backbone,
        'best_checkpoint': str(best_path),
        'history': history,
        'best_val_macro_f1': best_f1,
        'val_metrics': {k: v for k, v in val_metrics.items() if k not in {'y_true', 'y_pred', 'y_prob'}},
        'test_metrics': {k: v for k, v in test_metrics.items() if k not in {'y_true', 'y_pred', 'y_prob'}},
        'test_y_true': test_metrics['y_true'],
        'test_y_pred': test_metrics['y_pred'],
        'test_y_prob': test_metrics['y_prob'],
    }

In [ ]:
# Cell 8 — Train selected backbones
summaries = []
trained_models = {}
for backbone in BACKBONES:
    model, summary = train_one_backbone(backbone)
    summaries.append(summary)
    trained_models[backbone] = model

leaderboard = pd.DataFrame([
    {
        'backbone': s['backbone'],
        'val_macro_f1': s['val_metrics']['macro_f1'],
        'val_accuracy': s['val_metrics']['accuracy'],
        'test_macro_f1': s['test_metrics']['macro_f1'],
        'test_accuracy': s['test_metrics']['accuracy'],
        'checkpoint': s['best_checkpoint'],
    }
    for s in summaries
]).sort_values(['test_macro_f1', 'val_macro_f1'], ascending=False)
leaderboard_path = OUT_ROOT / 'cabin_exp_020a_leaderboard.csv'
leaderboard.to_csv(leaderboard_path, index=False)
display(leaderboard)
BEST_BACKBONE = leaderboard.iloc[0]['backbone']
print('BEST_BACKBONE:', BEST_BACKBONE)

In [ ]:
# Cell 9 — Evaluation plots and reports for the best model
best_summary = next(s for s in summaries if s['backbone'] == BEST_BACKBONE)
y_true = best_summary['test_y_true']
y_pred = best_summary['test_y_pred']

report_dict = classification_report(y_true, y_pred, target_names=LABELS, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_path = OUT_ROOT / 'cabin_exp_020a_test_classification_report.csv'
report_df.to_csv(report_path)
display(report_df)

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=LABELS, yticklabels=LABELS, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'{EXP_ID} Confusion Matrix — {BEST_BACKBONE}')
cm_path = OUT_ROOT / 'cabin_exp_020a_confusion_matrix.png'
plt.tight_layout()
plt.savefig(cm_path, dpi=180)
plt.show()
print('confusion matrix:', cm_path)

In [ ]:
# Cell 10 — Sample predictions and Grad-CAM-lite evidence previews
# This produces inspection images, not a final localization model.
import torch.nn.functional as F

INV_NORM = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225],
)

def predict_image(model, path):
    img = Image.open(path).convert('RGB')
    tensor = eval_tfms(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    return ID_TO_LABEL[pred_id], float(probs[pred_id]), probs

preview_dir = OUT_ROOT / 'previews'
preview_dir.mkdir(parents=True, exist_ok=True)
best_model = trained_models[BEST_BACKBONE]
best_model.eval()

sample_df = pd.concat([
    test_df[test_df['label'] == 'driver_cabin_visible'].sample(min(12, (test_df['label'] == 'driver_cabin_visible').sum()), random_state=SEED),
    test_df[test_df['label'] == 'not_cabin_view'].sample(min(12, (test_df['label'] == 'not_cabin_view').sum()), random_state=SEED),
])
preview_rows = []
for i, row in enumerate(sample_df.itertuples(index=False)):
    pred, conf, probs = predict_image(best_model, row.path)
    img = Image.open(row.path).convert('RGB').resize((320, 240))
    arr = np.array(img)
    canvas = np.zeros((300, 320, 3), dtype=np.uint8) + 255
    canvas[:240] = arr
    cv2.putText(canvas, f'true={row.label}', (8, 262), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
    cv2.putText(canvas, f'pred={pred} {conf:.3f}', (8, 285), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,0), 1, cv2.LINE_AA)
    out_path = preview_dir / f'preview_{i:03d}_{row.label}_pred_{pred}.jpg'
    cv2.imwrite(str(out_path), cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    preview_rows.append({'path': row.path, 'true': row.label, 'pred': pred, 'confidence': conf, 'preview': str(out_path)})

preview_df = pd.DataFrame(preview_rows)
preview_csv = OUT_ROOT / 'cabin_exp_020a_preview_predictions.csv'
preview_df.to_csv(preview_csv, index=False)
display(preview_df.head(20))
print('preview_dir:', preview_dir)

In [ ]:
# Cell 11 — Optional local demo video smoke inference
# Looks for Test/video_*.mp4 in Drive and samples frames. This is not accuracy evaluation.
TEST_VIDEO_DIRS = [
    PROJECT_ROOT / 'Test',
    PROJECT_ROOT / 'test',
]
SMOKE_EVERY_N_FRAMES = 25
SMOKE_MAX_FRAMES_PER_VIDEO = 80

def find_test_videos():
    videos = []
    for d in TEST_VIDEO_DIRS:
        if d.exists():
            videos.extend(sorted(d.glob('video_*.mp4')))
            videos.extend(sorted(d.glob('*.mp4')))
    # dedupe
    out = []
    seen = set()
    for v in videos:
        if v not in seen:
            out.append(v); seen.add(v)
    return out

def frame_to_pil(frame_bgr):
    return Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

smoke_rows = []
smoke_dir = OUT_ROOT / 'local_video_smoke_previews'
smoke_dir.mkdir(parents=True, exist_ok=True)
for video_path in find_test_videos():
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print('could not open:', video_path)
        continue
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    sample_ids = list(range(0, frame_count, SMOKE_EVERY_N_FRAMES))[:SMOKE_MAX_FRAMES_PER_VIDEO]
    for frame_id in tqdm(sample_ids, desc=f'smoke {video_path.name}'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        ok, frame = cap.read()
        if not ok:
            continue
        img = frame_to_pil(frame)
        tensor = eval_tfms(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(best_model(tensor), dim=1).cpu().numpy()[0]
        pred_id = int(np.argmax(probs))
        pred = ID_TO_LABEL[pred_id]
        conf = float(probs[pred_id])
        smoke_rows.append({
            'video': video_path.name,
            'frame_id': frame_id,
            'time_s': round(frame_id / fps, 3) if fps else None,
            'prediction': pred,
            'confidence': conf,
            'prob_driver_cabin_visible': float(probs[LABEL_TO_ID['driver_cabin_visible']]),
            'prob_not_cabin_view': float(probs[LABEL_TO_ID['not_cabin_view']]),
        })
        if len([r for r in smoke_rows if r['video'] == video_path.name]) <= 8:
            preview = cv2.resize(frame, (480, 270))
            cv2.putText(preview, f'{video_path.name} f={frame_id}', (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 3, cv2.LINE_AA)
            cv2.putText(preview, f'{pred} {conf:.3f}', (8, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 3, cv2.LINE_AA)
            cv2.putText(preview, f'{video_path.name} f={frame_id}', (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0), 1, cv2.LINE_AA)
            cv2.putText(preview, f'{pred} {conf:.3f}', (8, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0,0,0), 1, cv2.LINE_AA)
            out_path = smoke_dir / f'{video_path.stem}_frame_{frame_id:06d}.jpg'
            cv2.imwrite(str(out_path), preview, [int(cv2.IMWRITE_JPEG_QUALITY), 88])
    cap.release()

smoke_df = pd.DataFrame(smoke_rows)
smoke_csv = OUT_ROOT / 'cabin_exp_020a_local_video_smoke.csv'
smoke_df.to_csv(smoke_csv, index=False)
if len(smoke_df):
    display(smoke_df.groupby(['video', 'prediction']).agg(count=('prediction','size'), mean_conf=('confidence','mean')).reset_index())
else:
    print('No local demo videos found in Drive. Smoke inference skipped.')
print('smoke_csv:', smoke_csv)

In [ ]:
# Cell 12 — Save summary JSON, label map, and markdown report
summary = {
    'experiment_id': EXP_ID,
    'created_at_utc': now_utc(),
    'task': 'cabin_driver_view_binary_classifier',
    'labels': LABELS,
    'label_to_id': LABEL_TO_ID,
    'best_backbone': BEST_BACKBONE,
    'leaderboard': leaderboard.to_dict(orient='records'),
    'best_checkpoint': str(leaderboard.iloc[0]['checkpoint']),
    'dataset': {
        'metadata_csv': str(metadata_path),
        'split_csv': str(split_path),
        'train_rows': int(len(train_df)),
        'val_rows': int(len(val_df)),
        'test_rows': int(len(test_df)),
        'sources': split_df.groupby(['label', 'source']).size().reset_index(name='count').to_dict(orient='records'),
        'state_farm_source': 'https://www.kaggle.com/competitions/state-farm-distracted-driver-detection/data',
        'auc_optional_source': 'https://heshameraqi.github.io/distraction_detection',
        'negative_policy': 'manual_not_cabin_view_folder_or_existing_BDD100K_images',
    },
    'metrics': {
        'test_classification_report_csv': str(report_path),
        'confusion_matrix_png': str(cm_path),
        'leaderboard_csv': str(leaderboard_path),
    },
    'ftr_mapping': {
        'driver_cabin_visible': 'enables_downstream_sofor_eylemi_analysis',
        'not_cabin_view': 'no_cabin_driver_event_written',
    },
    'limitations': [
        'State Farm is mostly in-cabin camera imagery; target demo is outside-camera side/front windshield view.',
        '020A is a visibility/view gate, not final phone/smoking/seatbelt/passenger decision.',
        'Negative class quality strongly affects deployment behavior; manual negative review is required.',
    ],
}
summary_path = OUT_ROOT / 'cabin_exp_020a_summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

label_map_path = CKPT_ROOT / 'cabin_exp_020a_label_map.json'
label_map_path.write_text(json.dumps({'labels': LABELS, 'label_to_id': LABEL_TO_ID}, indent=2, ensure_ascii=False), encoding='utf-8')

report_md = REPORT_ROOT / 'cabin_exp_020a_driver_view_baseline.md'
report_md.write_text(f'''# CABIN-EXP-020A Driver/Cabin View Baseline

Bu rapor Colab notebook tarafindan uretilmistir.

## Amac

Heuristik `CABIN-EXP-012` reddedildikten sonra model-first cabin/driver gate baseline'i kurmak.

## Veri

* Pozitif: State Farm Distracted Driver Detection, driver/cabin gorunur goruntuler.
* Negatif: manuel `not_cabin_view` klasoru veya mevcut BDD100K yol/dis ortam goruntuleri.

## En Iyi Model

* Backbone: `{BEST_BACKBONE}`
* Checkpoint: `{leaderboard.iloc[0]['checkpoint']}`

## Leaderboard

{leaderboard.to_markdown(index=False)}

## Sinirlar

* Bu model eylem karari vermez; yalniz downstream driver action specialist modellerinin calisip calismayacagini belirleyen gate'tir.
* State Farm arac ici kamera domain'idir; dis kamera/yan cam domain farki raporda acik yazilmalidir.
* Lokal video smoke sonucu ground-truth dogruluk degildir.

## Sonraki Adim

`CABIN-EXP-020B` driver action classifier: `telefonla_konusma`, `su_icme`, `arkaya_bakma`, `etrafa_bakinma` mapping'i.
''', encoding='utf-8')

print('summary:', summary_path)
print('label_map:', label_map_path)
print('report:', report_md)